In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==============================================================================
# PROBLEM 6: Symbolic Z-Transform and Interactive Pole-Zero / ROC Visualization
# Signal: x[n] = A * r^n * cos(omega * n + phi) * u[n]
# ==============================================================================

explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>Solution Overview</b><br>
* <b>Signal:</b> x[n] = A * r^n * cos(ω * n + φ) * u[n] (0 < r < 1)<br>
* <b>Z-Transform Derivation:</b> X(z) = Az * [z*cos(φ) - r*cos(ω - φ)] / (z^2 - 2rz*cos(ω) + r^2)<br>
* <b>Poles & Zeros:</b> Simple complex conjugate poles at p₁,₂ = r * e^{±jω}, Zeros at z = 0 and z = r*cos(ω - φ)/cos(φ).<br>
* <b>ROC (Region of Convergence):</b> |z| > r.<br>
* <b>Note:</b> Use the sliders below to dynamically change r, ω, φ, and amplitude A to observe pole/zero movements and ROC changes.
</div>
"""
display(widgets.HTML(explanation_text))

out = widgets.Output()

n_samples = 30
n_vec = np.arange(n_samples)

def plot_problem_6(A_val, r_val, omega_val, phi_val):
    with out:
        clear_output(wait=True)
        
        fig, (ax_pz, ax_time) = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [1, 2]})
        plt.subplots_adjust(wspace=0.25)

        # --- 1. Pole-Zero Map & ROC ---
        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-2.0, 2.0)
        ax_pz.set_ylim(-2.0, 2.0)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)

        roc_radius = abs(r_val)

        # Proper ROC Shading for |z| > r using a grid mask
        x_vals = np.linspace(-2.5, 2.5, 400)
        y_vals = np.linspace(-2.5, 2.5, 400)
        X, Y = np.meshgrid(x_vals, y_vals)
        Z_dist = np.sqrt(X**2 + Y**2)
        roc_mask = Z_dist > roc_radius

        ax_pz.imshow(roc_mask, extent=(-2.5, 2.5, -2.5, 2.5), origin='lower', cmap='Greens', alpha=0.25, zorder=0)

        # ROC Boundary Circle
        theta = np.linspace(0, 2*np.pi, 200)
        if roc_radius > 0:
            ax_pz.plot(roc_radius * np.cos(theta), roc_radius * np.sin(theta), 'g:', linewidth=2, label=f'ROC Boundary (|z| = r)')

        # Unit Circle reference
        ax_pz.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5)

        # Zeros calculation (z = 0 and z = r*cos(omega - phi) / cos(phi))
        cos_phi = np.cos(phi_val)
        if abs(cos_phi) > 1e-5:
            z_zero2 = r_val * np.cos(omega_val - phi_val) / cos_phi
            zeros_x = [0, z_zero2]
            zeros_y = [0, 0]
        else:
            zeros_x = [0]
            zeros_y = [0]

        ax_pz.scatter(zeros_x, zeros_y, s=120, facecolors='none', edgecolors='b', linewidths=2, marker='o')

        # Simple complex conjugate poles at p = r * e^{±j*omega}
        p1_real = r_val * np.cos(omega_val)
        p1_imag = r_val * np.sin(omega_val)
        
        ax_pz.scatter([p1_real], [p1_imag], s=140, color='purple', marker='x', linewidths=3)
        ax_pz.scatter([p1_real], [-p1_imag], s=140, color='purple', marker='x', linewidths=3)

        stability = "Stable" if roc_radius < 1 else "Unstable"
        ax_pz.set_title(f'Pole-Zero Map & ROC ({stability}, |z| > {roc_radius:.2f})', fontsize=10, fontweight='bold')
        ax_pz.set_xlabel('Real Part', fontsize=9)
        ax_pz.set_ylabel('Imaginary Part', fontsize=9)

        # Legend handles in a single straight line
        unit_circle_handle = plt.Line2D([0], [0], color='k', linestyle='--', alpha=0.5, label='Unit Circle')
        pole_handle = plt.Line2D([0], [0], marker='x', color='purple', markersize=8, markeredgewidth=3, linestyle='None', label='Complex Poles (p₁, p₂)')
        zero_handle = plt.Line2D([0], [0], marker='o', markerfacecolor='none', markeredgecolor='b', markersize=8, markeredgewidth=2, linestyle='None', label='Zeros (z=0, root)')

        ax_pz.legend(handles=[unit_circle_handle, pole_handle, zero_handle], loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=8)

        # --- 2. Time Domain Plot ---
        x_n_vals = A_val * (r_val**n_vec) * np.cos(omega_val * n_vec + phi_val)

        ax_time.stem(n_vec, x_n_vals, linefmt='r-', markerfmt='ro', basefmt='k-')
        ax_time.set_title('Temporal Evolution: x[n] = A r^n cos(ω n + φ)u[n]', fontsize=10, fontweight='bold')
        ax_time.set_xlabel('Time index n', fontsize=9)
        ax_time.set_ylabel('x[n]', fontsize=9)
        ax_time.set_xlim(-1, n_samples)
        
        max_abs_val = np.max(np.abs(x_n_vals))
        y_limit = max(1.2, min(max_abs_val * 1.25, 20.0))
        ax_time.set_ylim(-y_limit, y_limit)
        ax_time.grid(True, linestyle=':', alpha=0.7)

        plt.show()

# Sliders for parameters
A_slider = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='A:', style={'description_width': 'initial'})
r_slider = widgets.FloatSlider(value=0.8, min=0.1, max=0.99, step=0.01, description='r:', style={'description_width': 'initial'})
omega_slider = widgets.FloatSlider(value=0.6, min=0.1, max=np.pi, step=0.01, description='Omega:', style={'description_width': 'initial'})
phi_slider = widgets.FloatSlider(value=0.0, min=-np.pi, max=np.pi, step=0.01, description='Phi:', style={'description_width': 'initial'})

plot_problem_6(A_slider.value, r_slider.value, omega_slider.value, phi_slider.value)

interactive_plot = widgets.interactive(plot_problem_6, A_val=A_slider, r_val=r_slider, omega_val=omega_slider, phi_val=phi_slider)
display(widgets.VBox([interactive_plot, out]))